# Animation Skill — Kaggle GPU Renderer

1. In Notebook Settings, select **GPU** (P100/T4 when available) and enable Internet.
2. Add the SheepIt export ZIP as a Kaggle Dataset input.
3. Run all cells. For GPU acceleration, the ZIP must contain Cycles scenes. The worker reports the actual device and falls back to CPU if no compatible GPU is exposed.

This notebook renders Blender frames and assembles a silent MP4. Narration/music/captions still need the normal FFmpeg finishing stage. Kaggle free GPU availability and quotas are controlled by Kaggle.

In [ ]:
!apt-get update -qq && apt-get install -y blender ffmpeg >/dev/null
!blender --version | head -n 2
!ffmpeg -version | head -n 1

In [ ]:
import glob, json, os, shutil, subprocess, zipfile
from pathlib import Path

archives = sorted(glob.glob('/kaggle/input/**/*.zip', recursive=True))
if not archives:
    raise FileNotFoundError('Attach the SheepIt/Kaggle package ZIP as a Kaggle Dataset input first.')
package_zip = Path(archives[0])
work = Path('/kaggle/working/animation-render')
if work.exists():
    shutil.rmtree(work)
work.mkdir(parents=True)
with zipfile.ZipFile(package_zip) as zf:
    zf.extractall(work)
manifest_path = next(work.rglob('manifest.json'))
manifest = json.loads(manifest_path.read_text())
worker = next(work.rglob('kaggle_gpu_render.py'))
blender = shutil.which('blender')
ffmpeg = shutil.which('ffmpeg')
print('Package:', package_zip)
print('Shots:', len(manifest['shots']))
print('Blender:', blender)
print('Worker:', worker)

In [ ]:
segments = []
for index, shot in enumerate(manifest['shots'], 1):
    blend = (manifest_path.parent / shot['blend']).resolve()
    frame_dir = work / ('frames_%03d' % index)
    frame_dir.mkdir(parents=True, exist_ok=True)
    command = [blender, '-b', str(blend), '--python', str(worker), '--', '--output-dir', str(frame_dir)]
    print('Rendering', shot['id'], 'with', shot['engine'], shot['resolution'], shot['fps'], 'FPS')
    subprocess.run(command, check=True)
    segment = work / ('segment_%03d.mp4' % index)
    subprocess.run([ffmpeg, '-y', '-framerate', str(shot['fps']), '-i', str(frame_dir / 'frame_%04d.png'), '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '18', '-pix_fmt', 'yuv420p', str(segment)], check=True)
    segments.append(segment)
concat = work / 'concat.txt'
concat.write_text(''.join("file '%s'\n" % p.as_posix() for p in segments))
output = Path('/kaggle/working/animation-final-silent.mp4')
subprocess.run([ffmpeg, '-y', '-f', 'concat', '-safe', '0', '-i', str(concat), '-c', 'copy', '-movflags', '+faststart', str(output)], check=True)
media = manifest.get('media', {})
video_input = output
if media.get('captions'):
    captions = (manifest_path.parent / media['captions']).resolve()
    captioned = work / 'animation-captioned.mp4'
    escaped = str(captions).replace('\\\\', '\\\\\\\\').replace(':', '\\\\:').replace("'", "\\\\'")
    subprocess.run([ffmpeg, '-y', '-i', str(video_input), '-vf', "subtitles=filename='%s'" % escaped, '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '18', '-pix_fmt', 'yuv420p', '-an', str(captioned)], check=True)
    video_input = captioned
voice = (manifest_path.parent / media['voice']).resolve() if media.get('voice') else None
music = (manifest_path.parent / media['music']).resolve() if media.get('music') else None
if voice or music:
    muxed = Path('/kaggle/working/animation-final.mp4')
    cmd = [ffmpeg, '-y', '-i', str(video_input)]
    if voice:
        cmd += ['-i', str(voice)]
    if music:
        cmd += ['-stream_loop', '-1', '-i', str(music)]
    if voice and music:
        cmd += ['-filter_complex', '[1:a]aresample=48000,apad[voice];[2:a]volume=0.18,aresample=48000,apad[music];[voice][music]amix=inputs=2:duration=longest:dropout_transition=2[aout]', '-map', '0:v:0', '-map', '[aout]']
    elif voice:
        cmd += ['-filter_complex', '[1:a]aresample=48000,apad[aout]', '-map', '0:v:0', '-map', '[aout]']
    else:
        cmd += ['-filter_complex', '[1:a]volume=0.18,aresample=48000,apad[aout]', '-map', '0:v:0', '-map', '[aout]']
    cmd += ['-c:v', 'copy', '-c:a', 'aac', '-b:a', '192k', '-shortest', '-movflags', '+faststart', str(muxed)]
    subprocess.run(cmd, check=True)
    video_input = muxed
print('OUTPUT_MP4:', video_input, 'bytes:', video_input.stat().st_size)


## Output
The final MP4 is saved at the printed path under `/kaggle/working/`. If narration/music/captions were included in the export package, this notebook adds them during FFmpeg finishing. Download it from the notebook's Output/Files panel. If the GPU cell reports CPU fallback, check the selected Kaggle accelerator and confirm that the scene was exported with `CYCLES`.
